<a href="https://colab.research.google.com/github/MehrabSiddique/converters/blob/main/convert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ✅ Step 1: Install required libraries
!pip install lxml

import os
import shutil
import zipfile
import base64
from lxml import etree
from google.colab import files

# ✅ Step 2: Prepare folders
shutil.rmtree("uploaded", ignore_errors=True)
shutil.rmtree("extracted", ignore_errors=True)
os.makedirs("uploaded", exist_ok=True)
os.makedirs("extracted", exist_ok=True)

# ✅ Step 3: Upload a DOCX file
print("📤 Upload a DOCX file to convert to Word-compatible XML")
uploaded = files.upload()

# Save file to folder
for filename in uploaded:
    input_path = os.path.join("uploaded", filename)
    shutil.move(filename, input_path)

# ✅ Step 4: Extract DOCX contents (it's a ZIP)
with zipfile.ZipFile(input_path, 'r') as zip_ref:
    zip_ref.extractall("extracted")

# ✅ Step 5: Build Flat OPC XML like Microsoft Word
def build_flat_opc_package():
    NSMAP = {None: "http://schemas.microsoft.com/office/2006/xmlPackage"}
    pkg = etree.Element("{http://schemas.microsoft.com/office/2006/xmlPackage}package", nsmap=NSMAP)

    for root, dirs, files_in_dir in os.walk("extracted"):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, "extracted").replace("\\", "/")

            with open(full_path, "rb") as f:
                content = f.read()

            # Determine content type
            if file.endswith(".rels"):
                content_type = "application/vnd.openxmlformats-package.relationships+xml"
            elif file.endswith(".xml"):
                content_type = "application/xml"
            else:
                content_type = "application/octet-stream"

            part = etree.SubElement(pkg, "{http://schemas.microsoft.com/office/2006/xmlPackage}part",
                                    name="/" + rel_path,
                                    contentType=content_type,
                                    padding="512")

            if content_type.startswith("application/xml") or content_type.endswith("relationships+xml"):
                try:
                    xml = etree.fromstring(content)
                    xmlData = etree.SubElement(part, "{http://schemas.microsoft.com/office/2006/xmlPackage}xmlData")
                    xmlData.append(xml)
                except etree.XMLSyntaxError:
                    binaryData = etree.SubElement(part, "{http://schemas.microsoft.com/office/2006/xmlPackage}binaryData")
                    binaryData.text = base64.b64encode(content).decode("utf-8")
            else:
                binaryData = etree.SubElement(part, "{http://schemas.microsoft.com/office/2006/xmlPackage}binaryData")
                binaryData.text = base64.b64encode(content).decode("utf-8")

    return pkg

# ✅ Step 6: Generate output XML
opc_tree = build_flat_opc_package()
output_path = "flat_opc_converted.xml"

with open(output_path, "wb") as f:
    f.write(b'<?mso-application progid="Word.Document"?>\n')
    f.write(etree.tostring(opc_tree, pretty_print=True, encoding="utf-8"))

# ✅ Step 7: Download the file
print("✅ Conversion complete! Download your XML below.")
files.download(output_path)

# Optional: Cleanup
shutil.rmtree("uploaded", ignore_errors=True)
shutil.rmtree("extracted", ignore_errors=True)


📤 Upload a DOCX file to convert to Word-compatible XML


Saving research-experience-evidence-template.docx to research-experience-evidence-template.docx
✅ Conversion complete! Download your XML below.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# Install required packages
!pip install lxml

import os
import shutil
import zipfile
import base64
from lxml import etree
from google.colab import files

# Clear workspace
shutil.rmtree("xml_input", ignore_errors=True)
shutil.rmtree("docx_folder", ignore_errors=True)
os.makedirs("xml_input", exist_ok=True)
os.makedirs("docx_folder", exist_ok=True)

# Upload Flat OPC XML file
print("📤 Upload your Flat OPC XML file:")
uploaded = files.upload()

# Save the uploaded file
for fname in uploaded:
    xml_path = os.path.join("xml_input", fname)
    shutil.move(fname, xml_path)

# Parse and rebuild .docx structure
def rebuild_docx_from_flat_opc(xml_file, output_dir):
    ns = {"pkg": "http://schemas.microsoft.com/office/2006/xmlPackage"}
    tree = etree.parse(xml_file)
    root = tree.getroot()

    for part in root.findall("pkg:part", ns):
        name = part.attrib["name"].lstrip("/")
        part_path = os.path.join(output_dir, name)
        os.makedirs(os.path.dirname(part_path), exist_ok=True)

        # Check for XML data
        xml_data = part.find("pkg:xmlData", ns)
        if xml_data is not None and len(xml_data):
            content = etree.tostring(xml_data[0], encoding="utf-8", xml_declaration=True)
            with open(part_path, "wb") as f:
                f.write(content)
        else:
            # Assume binary
            bin_data = part.find("pkg:binaryData", ns)
            if bin_data is not None:
                content = base64.b64decode(bin_data.text)
                with open(part_path, "wb") as f:
                    f.write(content)

# Extract parts
rebuild_docx_from_flat_opc(xml_path, "docx_folder")

# Create .docx file
def zip_docx_folder(source_folder, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as docx_zip:
        for root, dirs, files_in_dir in os.walk(source_folder):
            for file in files_in_dir:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, source_folder).replace("\\", "/")
                docx_zip.write(full_path, rel_path)

output_docx_path = "output.docx"
zip_docx_folder("docx_folder", output_docx_path)

# Download .docx
print("✅ Download your restored .docx file:")
files.download(output_docx_path)


📤 Upload your Flat OPC XML file:


Saving flat_opc_converted.xml to flat_opc_converted (1).xml
✅ Download your restored .docx file:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>